In [1]:
import pandas as pd

file_path = '../../hse-data/data/OAHP_MasterDataCatalogueVersion0002(Indicators).csv'

df = pd.read_csv(file_path, skiprows=[0, 2])
df.columns


Index(['dcterms:title', 'dct:identifier', 'hwbp:Rationale', 'skos:definition',
       'hwbp:indicatorType', 'hwbp:status', 'hwbp:disaggregation',
       'hwbp:numeratorDataElement',
       'hwbp:numeratorSource [a dct:dataset [a dct:publisher [a dcterms:source]]',
       'hwbp:denominatorDataElement',
       'hwbp:denominatorSource [a dct:dataset [a dct:publisher [a dcterms:source]]',
       'hwbp:methodology', 'dcterms:accrualPeriodicity ', 'hwbp:reportStyle',
       'healthdcatap:healthTheme', 'healthdcatap:healthTheme.1',
       'hwbp:highLowGuidance', 'hwbp:measurementLimitations',
       'hwbp:validityGuidance', 'hwbp:importance',
       'dct:provenance [a dct:ProvenanceStatement; rdfs:label ]', 'skos:note',
       'skos:historyNote', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25',
       'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29',
       'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33',
       'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36'],
      dtype='objec

In [2]:
# remove empty rows
df = df.dropna(how='all')

# print number of rows and columns
print(df.shape)

# print first 5 rows
print(df.head(5))

(138, 37)
                                       dcterms:title     dct:identifier  \
0  Number and proportion of population aged 65 ye...  SAP2022T1T1ASAC01   
1                Number of homeless persons aged 65+                NaN   
2                      Number of Travellers aged 65+           F9026C01   
3  Percentage of older adults (65+) who are in re...          HU015A_04   
4  Deprivation distribution by 8-point Pobal HP i...                NaN   

   hwbp:Rationale                                    skos:definition  \
0             NaN  The total number and percentage of people in t...   
1             NaN  The number of people aged 65 years and over wh...   
2             NaN  The number of people from the Traveller commun...   
3             NaN  The percentage of adults aged 65 years or over...   
4             NaN  How adults aged 65 years and over are spread a...   

   hwbp:indicatorType        hwbp:status hwbp:disaggregation  \
0                 NaN  Include in Delphi  

## The following code is to fill in / recreate identifiers for rows that meet the these three cases:
1. `Indicator ID` column is blank
2. `Indicator ID` column is a text
3. `Indicator ID` column contains multiple ids separated by ', '

## Data Cleaning Step: Remove Special Characters from Indicator ID

Before proceeding with ID recreation, we need to clean the "Indicator ID" column to remove special characters, unwanted whitespace, and other formatting issues.

In [3]:
import re

# Function to clean special characters from Indicator ID
def clean_indicator_id(id_value):
    """
    Clean special characters from Indicator ID column
    - Remove special characters like �, quotes, newlines
    - Keep only alphanumeric, underscore, hyphen, colon
    - Strip whitespace
    """
    if pd.isna(id_value):
        return id_value  # Keep NaN as is
    
    # Convert to string if not already
    id_str = str(id_value)
    
    # Remove special characters, keep only: letters, numbers, underscore, hyphen, colon, period
    cleaned = re.sub(r'[^\w\-:.]', '', id_str)
    
    # Remove extra spaces and strip
    cleaned = re.sub(r'\s+', '', cleaned)
    
    # Return empty string as NaN for further processing
    return cleaned if cleaned else None

# Show sample of problematic IDs before cleaning
print("=== Sample Indicator IDs BEFORE cleaning ===")
sample_ids = df['dct:identifier'].head(20)
for i, id_val in enumerate(sample_ids):
    print(f"Row {i}: '{id_val}' (type: {type(id_val)})")

print("\n" + "="*50)

# Apply cleaning to the Indicator ID column
df['dct:identifier'] = df['dct:identifier'].apply(clean_indicator_id)

print("\n=== Sample Indicator IDs AFTER cleaning ===")
sample_ids_cleaned = df['dct:identifier'].head(20)
for i, id_val in enumerate(sample_ids_cleaned):
    print(f"Row {i}: '{id_val}' (type: {type(id_val)})")

# Show statistics about the cleaning
print(f"\n=== Cleaning Statistics ===")
print(f"Total rows: {len(df)}")
print(f"Empty/NaN IDs after cleaning: {df['dct:identifier'].isna().sum()}")
print(f"Non-empty IDs after cleaning: {df['dct:identifier'].notna().sum()}")

# Show unique patterns of special characters that were found (for debugging)
print(f"\n=== Unique ID patterns after cleaning (first 10) ===")
unique_ids = df['dct:identifier'].dropna().unique()[:10]
for uid in unique_ids:
    print(f"'{uid}'")

=== Sample Indicator IDs BEFORE cleaning ===
Row 0: 'SAP2022T1T1ASAC01' (type: <class 'str'>)
Row 1: 'nan' (type: <class 'float'>)
Row 2: 'F9026C01' (type: <class 'str'>)
Row 3: 'HU015A_04' (type: <class 'str'>)
Row 4: 'nan' (type: <class 'float'>)
Row 5: 'F9026C01' (type: <class 'str'>)
Row 6: 'F4043C02' (type: <class 'str'>)
Row 7: 'PH401' (type: <class 'str'>)
Row 8: 'HU010' (type: <class 'str'>)
Row 9: 'HU012' (type: <class 'str'>)
Row 10: 'FY009C02' (type: <class 'str'>)
Row 11: 'ICD10_09' (type: <class 'str'>)
Row 12: 'PH501, PH502' (type: <class 'str'>)
Row 13: 'MHcesd_capi_sf' (type: <class 'str'>)
Row 14: 'HU008' (type: <class 'str'>)
Row 15: 'F3084C02' (type: <class 'str'>)
Row 16: 'F3084C03' (type: <class 'str'>)
Row 17: 'HU007' (type: <class 'str'>)
Row 18: 'F4001C01' (type: <class 'str'>)
Row 19: 'WBD08C01' (type: <class 'str'>)


=== Sample Indicator IDs AFTER cleaning ===
Row 0: 'SAP2022T1T1ASAC01' (type: <class 'str'>)
Row 1: 'nan' (type: <class 'float'>)
Row 2: 'F9026C

In [4]:
id_template = 'hse'
id_length = 8  # Total length should be 8 characters

def create_padded_id(index):
    """Create an 8-character ID with zero-padding"""
    # Calculate how many digits we need for padding
    # 'hse' = 3 characters, so we need 5 more digits
    number_part = str(index).zfill(5)  # Zero-pad to 5 digits
    return f"{id_template}{number_part}"

# case 1: Handle rows with NaN identifiers (after cleaning)

# get rows that have nan identifiers
nan_identifiers = df[df['dct:identifier'].isna()]
print(f"Number of rows with NaN identifiers: {len(nan_identifiers)}")

# Recreate IDs for NaN entries
for index, row in nan_identifiers.iterrows():
    row_dict = row.to_dict()
    print(f"Original ID at index {index}: {row_dict['dct:identifier']}")
    new_id = create_padded_id(index)
    df.at[index, 'dct:identifier'] = new_id
    # print row with new id
    print(f"New ID assigned: {df.at[index, 'dct:identifier']}")
    print("-" * 40)

print(f"\nCompleted case 1: {len(nan_identifiers)} IDs recreated")

# Show summary after case 1
print(f"\nSummary after cleaning and case 1:")
print(f"Total rows: {len(df)}")
print(f"Remaining NaN IDs: {df['dct:identifier'].isna().sum()}")
print(f"Non-empty IDs: {df['dct:identifier'].notna().sum()}")

# Show some examples of the newly created IDs
new_ids = df[df['dct:identifier'].str.startswith('hse0', na=False)]['dct:identifier'].head(5)
print(f"\nExamples of newly created IDs:")
for new_id in new_ids:
    print(f"  {new_id} (length: {len(new_id)})")

Number of rows with NaN identifiers: 12
Original ID at index 1: nan
New ID assigned: hse00001
----------------------------------------
Original ID at index 4: nan
New ID assigned: hse00004
----------------------------------------
Original ID at index 41: nan
New ID assigned: hse00041
----------------------------------------
Original ID at index 54: nan
New ID assigned: hse00054
----------------------------------------
Original ID at index 88: nan
New ID assigned: hse00088
----------------------------------------
Original ID at index 91: nan
New ID assigned: hse00091
----------------------------------------
Original ID at index 92: nan
New ID assigned: hse00092
----------------------------------------
Original ID at index 110: nan
New ID assigned: hse00110
----------------------------------------
Original ID at index 113: nan
New ID assigned: hse00113
----------------------------------------
Original ID at index 120: nan
New ID assigned: hse00120
----------------------------------------

In [5]:
# Case 2: Handle text-only identifiers (non-alphanumeric patterns)
# Case 3: Handle multiple IDs separated by commas or other delimiters - return the first id and put the original id in the column 'skos:note'

import re

def is_valid_id_pattern(id_str):
    """
    Check if ID follows expected pattern (alphanumeric with allowed special chars)
    Returns False for pure text descriptions
    """
    if pd.isna(id_str) or not id_str:
        return False
    
    # Convert to string
    id_str = str(id_str).strip()
    
    # Check if it's mostly text (more than 50% letters vs numbers/symbols)
    letter_count = len(re.findall(r'[a-zA-Z]', id_str))
    total_non_space = len(re.sub(r'\s', '', id_str))
    
    if total_non_space == 0:
        return False
        
    letter_ratio = letter_count / total_non_space
    
    # If it's mostly letters and has spaces, it's probably descriptive text
    if letter_ratio > 0.8 and ' ' in id_str:
        return False
    
    # If it contains multiple IDs (comma separated), needs processing
    if ',' in id_str and len(id_str.split(',')) > 1:
        return False
        
    return True

def process_multiple_ids(id_str, index):
    """
    Process IDs that contain multiple values separated by commas
    Returns the first valid ID or creates a new one
    """
    if ',' in str(id_str):
      id_split = id_str.split(',')
      return id_split[0].strip()  # Return first valid ID after stripping whitespace
    
    # If no valid ID found, create new one with 8-character format
    return create_padded_id(index)

# Find and process problematic IDs
print("=== Case 2: Handling text-only and invalid ID patterns ===")

# Find rows with invalid ID patterns
# invalid_pattern_mask = ~df['dct:identifier'].apply(is_valid_id_pattern)
# invalid_ids = df[invalid_pattern_mask]

multiple_ids_mask = df['dct:identifier'].str.contains(',', na=False)
multiple_ids = df[multiple_ids_mask]

print(f"Found {len(multiple_ids)} rows with invalid ID patterns:")

for index, row in multiple_ids.iterrows():
    original_id = row['dct:identifier']
    print(f"\nRow {index}: '{original_id}'")
    
    # Check if it contains multiple IDs
    if ',' in str(original_id):
        print("  -> Contains multiple IDs")
        new_id = process_multiple_ids(original_id, index)
        df.at[index, 'skos:note'] += f" Original id: {original_id}."  # Store original ID in 'skos:note'
        print(f"  -> Using first valid ID: '{new_id}'")
    else:
        # Pure text - create new ID with 8-character format
        print("  -> Pure text description, creating new ID")
        new_id = create_padded_id(index)
        print(f"  -> New ID: '{new_id}' (length: {len(new_id)})")
    
    # Update the DataFrame
    df.at[index, 'dct:identifier'] = new_id

print(f"\n=== Summary after all cleaning cases ===")
print(f"Total rows: {len(df)}")
print(f"Remaining NaN IDs: {df['dct:identifier'].isna().sum()}")
print(f"Non-empty IDs: {df['dct:identifier'].notna().sum()}")

# Show final sample of IDs
print(f"\n=== Final sample of cleaned IDs ===")
sample_final = df['dct:identifier'].head(10)
for i, id_val in enumerate(sample_final):
    print(f"Row {i}: '{id_val}' (length: {len(id_val)})")

=== Case 2: Handling text-only and invalid ID patterns ===
Found 0 rows with invalid ID patterns:

=== Summary after all cleaning cases ===
Total rows: 138
Remaining NaN IDs: 0
Non-empty IDs: 138

=== Final sample of cleaned IDs ===
Row 0: 'SAP2022T1T1ASAC01' (length: 17)
Row 1: 'hse00001' (length: 8)
Row 2: 'F9026C01' (length: 8)
Row 3: 'HU015A_04' (length: 9)
Row 4: 'hse00004' (length: 8)
Row 5: 'F9026C01' (length: 8)
Row 6: 'F4043C02' (length: 8)
Row 7: 'PH401' (length: 5)
Row 8: 'HU010' (length: 5)
Row 9: 'HU012' (length: 5)


In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.min_rows', None)   # important to avoid head/tail with "..."
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', None)

print(df['dct:identifier'])
mask = df['dct:identifier'].str.contains(',')
multiple_ids = df[mask]
# invalid_pattern_mask = ~df['dct:identifier'].apply(is_valid_id_pattern)
# invalid_ids = df[invalid_pattern_mask]
print(f'invalid_ids: {multiple_ids}')

# reset display settings
pd.reset_option('display.max_rows')  # Reset to default
pd.reset_option('display.min_rows')  # Reset to default
pd.reset_option('display.max_columns')  # Reset to default
pd.reset_option('display.width')  # Reset to default
pd.reset_option('display.max_colwidth')  # Reset to default

0                     SAP2022T1T1ASAC01
1                              hse00001
2                              F9026C01
3                             HU015A_04
4                              hse00004
5                              F9026C01
6                              F4043C02
7                                 PH401
8                                 HU010
9                                 HU012
10                             FY009C02
11                             ICD10_09
12                           PH501PH502
13                       MHcesd_capi_sf
14                                HU008
15                             F3084C02
16                             F3084C03
17                                HU007
18                             F4001C01
19                             WBD08C01
20                          DISlongterm
21                            HU015A_01
22                            HU015A_03
23                             PEC23C01
24                             F4055C01
